In [25]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split


# Data
### data

In [26]:
ROOT= Path.cwd().parents[0]
path = ROOT / "data/raw"
train_data = path / "train.csv"
test_data = path / "test.csv"

### Data loading

In [27]:
def load_data() ->pd.DataFrame:
    df_train = pd.read_csv(train_data)
    df_test =pd.read_csv(test_data)
    return df_train, df_test

### Data Normalization 

In [28]:
def data_normalization(df:pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["text"] = df["text"].str.strip()
    return df 

### Data preprocessing

In [29]:
def data_preproccing(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    # remove duplication
    df = df.drop_duplicates()
    # remove missing values
    df = df.dropna()
    return df

### Validate Data

In [30]:
def validate_data(df: pd.DataFrame) ->None:
    required_columns = {"text", "label"}
    if set(df.columns) != required_columns:
        raise ValueError(
            f"Columns are not in {required_columns}"
        )
    if df["text"].isna().any():
        raise ValueError(
            "Missing value found"
        )
    if df["text"].duplicated().any():
        raise ValueError(
            "data contains duplicated rows"
        )


In [31]:
def split_data(df:pd.DataFrame,
               test_size:float = 0.2,
               random_state:int = 42
               ) -> tuple[pd.Series, pd.Series, pd.Series, pd.Series]:
    train_data = df
    X = train_data["text"]
    y = train_data["label"]
    
    X_train, X_val, y_train, y_val = train_test_split(
        X,
        y,
        random_state=random_state,
        test_size=test_size,
        stratify = y)
    
    return X_train, X_val, y_train, y_val
    

### Prepare data

In [32]:
def prepare_data() -> pd.DataFrame:
    df_train, df_test= load_data()
    validate_data(df_train)
    validate_data(df_test)
    df = data_normalization(df_train)
    df = data_preproccing(df)
    validate_data(df)
    
    return df
    

In [33]:
print(prepare_data())

                                                    text            label
0                         I am still waiting on my card?     card_arrival
1      What can I do if my card still hasn't arrived ...     card_arrival
2      I have been waiting over a week. Is the card s...     card_arrival
3      Can I track my card while it is in the process...     card_arrival
4      How do I know if I will get my card, or if it ...     card_arrival
...                                                  ...              ...
9998              You provide support in what countries?  country_support
9999                  What countries are you supporting?  country_support
10000                What countries are getting support?  country_support
10001                     Are cards available in the EU?  country_support
10002                   Which countries are represented?  country_support

[9999 rows x 2 columns]


# Feature
### text.py1

In [34]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer


class TextVectorizer:
    
    def __init__(
        self,
        method: str,
        vocab_size:int,
        ngram_range:tuple[int, int],
        min_df:float,
    ) ->None:
        self.method = method
        self.vocab_size = vocab_size
        self.ngram_range = ngram_range
        self.min_df = min_df
        self.vectorizer = self._create_vectorizer()
        
    def _create_vectorizer(self) -> CountVectorizer | TfidfVectorizer:
        
        if self.method == "bow":
            return CountVectorizer(
                max_features = self.vocab_size,
                ngram_range = self.ngram_range,
                min_df = self.min_df
            )
        elif self.method =="tfidf":
            return TfidfVectorizer(
                max_features = self.vocab_size,
                ngram_range =self.ngram_range,
                min_df = self.min_df
            )
            
        else:
            raise ValueError(
                "Method should be either 'bow' or 'tfidf'"
            )
    
    
    def fit_transform(
        self,
        text):
        vectorizer = self.vectorizer
        return vectorizer.fit_transform(text)
    
    def transform(
        self,
        text,
    ) -> pd.Series:
        return self.vectorizer.transform(text)
    
    

# Tokenizer 

In [35]:
from tensorflow.keras.layers import TextVectorization


class SequenceTokenizer:
    
    def __init__(
        self,
        max_tokens:int,
        output_sequence_length:int,
        output_mode:str
    ):
        self.max_tokens = max_tokens
        self.output_sequence_length = output_sequence_length
        self.output_mode = output_mode
        self.vectorizer = self._build_tokenizer()
        
    def _build_tokenizer(self) -> None:
        return TextVectorization(
            max_tokens = self.max_tokens,
            output_sequence_length= self.output_sequence_length,
            output_mode = self.output_mode
        )
    
    def fit(
        self,
        text,
    ) -> None:
        self.vectorizer.adapt(text)
    
    def transform(
        self,
        text
    ):
        return self.vectorizer(text)     

# Models

### Classical models

In [36]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"

sys.path.insert(0, str(SRC_DIR))
from configs import load_config
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import ComplementNB, MultinomialNB
from sklearn.svm import LinearSVC


class ClassicalModelFactory:
    
    def __init__(
        self,
        )-> None:
        
        self.config = load_config(PROJECT_ROOT/"configs/model.yaml")["classical"]["models"]
        
    def create_model(
        self,
        method: str
        ):
        
        
        models = {
            "logistic_regression" :LogisticRegression,
            "complement_nb":ComplementNB,
            "multinomial_nb":MultinomialNB,
            "linear_svc": LinearSVC,
        }
        if method not in models:
            raise ValueError(
                f"method is not in {models.keys()}"
            )
        return models[method](**self.config[method])
            
            
    def train(self, method, X_train, y_train):
        model = self.create_model(method)
        return model.fit(X_train, y_train)
        
        
    def predict(self, model, X_val):
        return model.predict(X_val)
         
# df = prepare_data()    

# X_train, X_val, y_train, y_val = split_data(df)

# vectorizer = TextVectorizer("tfidf", 10_000, (1, 2), 1)
# X_train_vectorized = vectorizer.fit_transform(X_train)
# X_val_vectorized = vectorizer.transform(X_val)

# instant = ClassicalModelFactory()
# lr = instant.train("logistic_regression", X_train_vectorized, y_train)

# prediction = lr.predict(X_val_vectorized)

# from sklearn.metrics import f1_score

# print(f"Validation Macro F1: {f1_score(y_val, prediction, average='macro'):.4f}")

In [37]:
# from pathlib import Path

# print("Current directory:", Path.cwd())
# print("Project root:", Path.cwd().parent)
# print("SRC contents:", list((Path.cwd().parent / "src").iterdir()))

# Neural.py

In [38]:
import torch
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence


class EmbeddingBagMLP(nn.Module):
    def __init__(
        self,
        vocab_size:int,
        embedding_dim:int,
        hidden_dim:int,
        num_classes:int=77,
        dropout:float = 0.2,
        padding_idx:int = 0,
    ):
        self.padding_idx = padding_idx
        super().__init__()
        

        
        self.embedding = nn.Embedding(
            num_embeddings = vocab_size,
            embedding_dim = embedding_dim,
            padding_idx = padding_idx,
        )
        
        self.classifier = nn.Sequential(
            nn.Linear(
                in_features=embedding_dim,
                out_features=hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(
                in_features=hidden_dim,
                out_features = num_classes
            )
        )
        
    def forward(self, token_ids):
        
        embeddings = self.embedding(token_ids)
        
        mask = (token_ids != self.padding_idx).unsqueeze(-1)
        
        embeddings = mask * embeddings
        
        summed_embeddings = embeddings.sum(dim=1)
        
        lengths = mask.sum(dim=1).clamp(min=1)
        
        mean_embedding = summed_embeddings / lengths
        
        return self.classifier(mean_embedding)

In [39]:
class _RecurrentClassifier(nn.Module):
    
    def __init__(
        
        self,
        vocab_size: int,
        embedding_dim: int,
        hidden_dim: int,
        rnn_type: str,
        dropout: float = 0.2,
        num_classes:int = 77,
        num_layers: int = 1,
        nonlinearity: str = 'tanh',
        batch_first: bool = True,
        bidirectional: bool = False,
        padding_idx: int= 0
        
    ):
        super().__init__()
        
        self.rnn_type = rnn_type
        self.bidirectional = bidirectional
        self.padding_idx = padding_idx
        
        
        self.embedding = nn.Embedding(
            num_embeddings = vocab_size,
            embedding_dim = embedding_dim,
            padding_idx = padding_idx
        )
        recurrrent_dropout = dropout if num_layers > 1 else 0.0
        if rnn_type == "RNN":
            self.recurrent = nn.RNN(
                input_size = embedding_dim,
                hidden_size = hidden_dim,
                num_layers = num_layers,
                nonlinearity = nonlinearity,
                batch_first = batch_first,
                bidirectional=bidirectional,
                dropout=recurrrent_dropout,
                
            )
        elif rnn_type == "LSTM":
            self.recurrent = nn.LSTM(
                input_size = embedding_dim,
                hidden_size = hidden_dim,
                num_layers = num_layers,
                batch_first = batch_first,
                dropout= recurrrent_dropout,
                bidirectional=bidirectional
            ) 
        elif rnn_type == "GRU":
            self.recurrent = nn.GRU(
                input_size = embedding_dim,
                hidden_size = hidden_dim,
                num_layers = num_layers,
                batch_first = batch_first,
                dropout= recurrrent_dropout,
                bidirectional=bidirectional
            )
        else:
            raise ValueError(
                "Invalid input, Valid inputs ' rnn, lstm, gru '"
            )
            
        self.classifier = nn.Linear(
            hidden_dim * (2 if bidirectional else 1),
            num_classes
        )
    def forward(self, token_ids:torch.Tensor) -> torch.Tensor:
        embedding = self.embedding(token_ids)
        
        lengths = (token_ids != self.padding_idx).sum(dim=1)
        
        if (lengths == 0 ).any():
            raise ValueError(
                "one of sequences is totally zero do somthing Farid"
            )
            
        packed = pack_padded_sequence(
            embedding.
            lengths.cpu(),
            batch_first= True,
            enforce_sorted = False
        )
        _, hidden = self.recurrent(packed)
        if self.rnn_type == "LSTM":
            hidden = hidden[0]
            
        if self.bidirectioal:
            feature = torch.cat(
                (hidden[-2], hidden[-1]),
                dim = 1
            )
        else:
            feature = hidden[-1]
        
        return self.classifier(feature)
    
    
class RNNClassifier(_RecurrentClassifier):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, rnn_type="RNN", **kwargs)
        
class LSTMClassifier(_RecurrentClassifier):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, rnn_type="LSTM", **kwargs)
        
class GRUClassifier(_RecurrentClassifier):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, rnn_type="GRU", **kwargs)
        
        

# Trainer 

In [40]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"
sys.path.insert(0, str(SRC_DIR))

In [41]:
import pandas as pd
from features.text import TextVectorizer
from models.classical import ClassicalModel


class Trainer:
    
    def __init__(
        self,
        config: dict,
    ) -> None:
        self.config = config["classical"]
        
    def extract_features(
        self,
        X_train:pd.Series,
        X_val:pd.Series,
        method:str):
        
        vectorizer = TextVectorizer(
            self.config["features"],
            method
        )
        
        X_train_vectorized = vectorizer.fit_transform(X_train)
        X_val_vectorized = vectorizer.transform(X_val)
        
        return vectorizer, X_train_vectorized, X_val_vectorized
    
    
    def train_model(
        self,
        X_train_vectorized,
        y_train,
        X_val_vectorized,
        method,
    ):
        model = ClassicalModel(self.config["models"], method)
        
        model.fit(X_train_vectorized, y_train)
        predict = model.predict(X_val_vectorized)
        
        return predict
    
    


# Neural Trainer

In [42]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

SRC_ROOT = PROJECT_ROOT / "src"

sys.path.insert(0, str(SRC_ROOT))

In [43]:
from __future__ import annotations  # noqa: F404

import random

import numpy as np
import torch
from configs import load_config
from data.data import prepare_data, split_data
from evaluation.evaluator import ClassificationEvaluator
from features.tokenizer import SequenceTokenizer
from models.neural import EmbeddingBagMLP, GRUClassifier, LSTMClassifier, RNNClassifier
from sklearn.preprocessing import LabelEncoder
from torch import nn
from torch.utils.data import DataLoader, TensorDataset


In [44]:
PADDING_IDX = 0

RECURRENT_MODELS = {"rnn", "lstm", "gru"} 

In [45]:
class NeuralTrainer:
      
    def __init__(self,config_path: str | Path = "configs/model.yaml") ->None:
        self.config = load_config(config_path)["neural"] 
        
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        self.label_encoder = LabelEncoder()
        self.tokenizer = None
        
        self.model = None
        self.optimizer = None
        self.criterion = nn.CrossEntropyLoss()
        
        
        self.train_loader = None
        self.val_loader = None
        self.evaluator = ClassificationEvaluator()
        
        
    def prepare_datasets(self) -> None:
        train_df, _ = prepare_data()
        
        X_train, X_val, y_train, y_val = split_data(train_df)
        
        self.tokenizer = SequenceTokenizer(
            vocab_size = self.config["vocab_size"],
            sequence_length = self.config["sequence_length"],
        )
        
        self.tokenizer.fit(X_train)
        X_train = self.tokenizer.transform(X_train)
        X_val = self.tokenizer.transform(X_val)
        
        y_train_ids = self.label_encoder.fit_transform(y_train)
        y_val_ids = self.label_encoder.transform(y_val)
        
        
        train_dataset = TensorDataset(
            self._to_tensor(X_train),
            self._to_tensor(y_train_ids)
        )
        val_dataset = TensorDataset(
            self._to_tensor(X_val),
            self._to_tensor(y_val_ids)
        )
        loader_kwargs = {
            "pin_memory": self.device.type == "cuda",
            "batch_size": self.config["batch_size"]
        }
        self.train_loader = DataLoader(train_dataset, shuffle=True, **loader_kwargs)
        self.val_loader  = DataLoader(val_dataset, shuffle=False, **loader_kwargs)
        
    @staticmethod
    def _to_tensor(values)-> torch.Tensor:
        return torch.as_tensor(np.asarray(values), dtype=torch.long)
    
    
    def _set_seed(self) ->None:
        seed = self.config.get("seed", 42)
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        
        
        
    def build_model(self,model_name) ->nn.Module:
        model_classes = {
            "mlp_classifier":EmbeddingBagMLP,
            "rnn":RNNClassifier,
            "lstm":LSTMClassifier,
            "gru":GRUClassifier,
        }
        if model_name not in model_classes:
            raise ValueError(
                f"Unsupported model: {model_name}. Choose from {sorted(model_classes)}."
            )
        
        self._set_seed()
        
        model_kwargs = {
            
            "vocab_size": len(self.tokenizer.get_vocabulary()),
            "embedding_dim": self.config["embedding_dim"],
            "hidden_dim": self.config["hidden_dim"],
            "dropout":self.config["dropout"],
            "num_classes":len(self.label_encoder.classes_),
            "padding_idx":PADDING_IDX,
            
        }
        
        if model_name in RECURRENT_MODELS:
            model_kwargs["num_layers"] = self.config.get("num_layers", 1)
            model_kwargs["bidirectional"] = self.config.get("bidirectional", False)
            
        self.model = model_classes[model_name](** model_kwargs).to(self.device)
        self.optimizer = torch.optim.Adam(
            self.model.parameters(),
            lr = self.config["learning_rate"]
        )
        
        return self.model
            
        
    
    def _run_epoch(self, loader: DataLoader, train:bool) ->dict:
        
        self.model.train(train)
        
        
        total_loss = 0
        total_samples = 0
        all_labels, all_predictions = [], []
        
        grad_context = torch.enable_grad() if train else torch.inference_mode()
        
        
        with grad_context:
            
            for token_ids, labels in loader:
                
               token_ids = token_ids.to(self.device, non_blocking = True)
               labels = labels.to(self.device, non_blocking =True)
               
               if train:
                   self.optimizer.zero_grad(set_to_none=True)
                   
               logits = self.model(token_ids)
               loss = self.criterion(logits, labels)
               
               if train:
                    loss.backward()
                    
                    nn.utils.clip_grad_norm_(
                        self.model.parameters(),
                        max_norm = self.config.get("grad_clip", 1.0)
                    )
                    
                    self.optimizer.step()
                
            batch_size = labels.size(0)
            total_loss += loss * batch_size
            total_samples += batch_size
            
            
            all_labels.append(labels.cpu())
            all_predictions.append(logits.argmax(dim=1).cpu())
            
            
        y_true = torch.cat(all_labels).numpy()
        y_pred = torch.cat(all_predictions).numpy()
        
        
        return {
            "loss":total_loss / total_samples,
            "accuracy": float(y_true == y_pred).mean(),
            "y_true": y_true,
            "y_pred": y_pred
        }
                
    
    def train_epoch(self) ->dict:
        return self._run_epoch(self.train_loader, train=True)
    
    
    def evaluate(self) ->dict:
        result = self._run_epoch(self.val_loader, train=False)
        metrics = self.evaluator.compute_metrics(y_true= result["y_true"], y_pred = result["y_pred"])
        return {
            "loss": result["loss"],
            "accuracy": result["accuracy"],
            **metrics
        }
    
    
    def fit(self, model_name:str) -> list[dict]:
        
       if self.train_loader is None or self.val_loader is None:
           
           raise RuntimeError(
               "Call prepare_datasets() before fit()."
               )
       self.build_model(model_name)
       
       epochs = self.config["epochs"]
       patience = self.config.get("patience")
       
       history = []
       best_val_loss = float("inf")
       best_state = None
       epochs_without_improvement = 0
    
        
       for epoch in range(1, epochs + 1):
           
           train_metrics = self.train_epoch()
           val_metrics = self.evaluate()
           
           history.append({
               "epoch":epoch,
               "loss": train_metrics,
               "validation":val_metrics,
               
           })
           
           print(f"Epoch {epoch} / {epochs}-> train Loss {train_metrics["loss"]:.4f}")
           print(f"Epoch {epoch} / {epochs}-> train accuracy {train_metrics["accuracy"]:.4f}")
           print(f"Epoch {epoch} / {epochs}-> Validation Loss {val_metrics["loss"]:.4f}")
           print(f"Epoch {epoch} / {epochs}-> Validation Accuracy {val_metrics["accuracy"]:.4f}")
           
           if val_metrics["loss"] < best_val_loss:
               best_val_loss = val_metrics["loss"]
               
               best_state = {
                   name: tensor.detach().cpu().clone()
                   for name, tensor in self.model.state_dict.items()
               }
           else:
                epochs_without_improvement +=1
            
           if patience is not None and epochs_without_improvement >= patience:
               print(f"Early stopping: no improvement for {patience} epochs.")
               break
        
       if best_state is not None:
            return self.model.load_state_dict(best_state)
        
        
       return history
                
        
           
               
                
           
           
            
        

# Transformer

In [46]:
from __future__ import annotations

import math

import torch
from torch import nn


In [47]:
class PositionalEncoding(nn.Module):
    
    def __init__(
        self,
        embedding_dim:int,
        max_length:int) -> None:
        super().__init__()
        
        positions = torch.arange(max_length).unsqueeze(1)
        frequencies = torch.exp(
            torch.arange(0, embedding_dim, 2) * (-math.log(10000.0) / embedding_dim)
        )
        
        encoding = torch.zeros(max_length, embedding_dim)
        encoding[:, 0::2] = torch.sin(positions * frequencies)
        encoding[:, 1::2] = torch.cos(positions * frequencies[: embedding_dim // 2])
        
        
        self.register_buffer("encoder", encoding.unsqueeze(0), persistent=False)
        
    def forward(self, embeddings: torch.Tensor) ->torch.Tensor:
        return embeddings + self.encoding[:, :embeddings.size(1)]

In [ ]:
class TransformerClassifier(nn.Module):
    
    def __init__(
        self,
        vocab_size:int,
        embedding_dim:int,
        num_classes:int,
        max_length:int,
        num_heads:int,
        num_layers:int,
        feedforward_dim: int |None = None,
        dropout:float = 0.1,
        padding_idx: int = 0
        ) ->None:
        super().__init__()
        
        if num_heads < 1 or embedding_dim % num_heads != 0:
            raise ValueError(
                f"embedding_dim ({embedding_dim}) must be divisible by "
                f"num_heads ({num_heads})."
            )
            
        if num_layers < 1:
            raise ValueError("num_layers must be at least 1.")
        if max_length < 1:
            raise ValueError("max_length must be at least 1.")
        
        if feedforward_dim is None:
            feedforward_dim = 4 * embedding_dim
            
        self.embedding_dim = embedding_dim
        self.padding_idx = padding_idx
        self.max_length = max_length
        
        self.embedding = nn.Embedding(
            num_embeddings = vocab_size,
            embedding_dim = embedding_dim,
            padding_idx = padding_idx
        )
        
        self.positional_encoding = PositionalEncoding(
                            embedding_dim,
                            max_length)
        
        
        self.embedding_dropout = nn.Dropout(dropout)
        
        encoder_layer = nn.TransformerEncoderLayer(
           d_model = embedding_dim,
           nhead = num_heads,
           dim_feedforward = feedforward_dim,
           dropout = dropout,
           activation = "gelu",
           batch_first = True,
           norm_first = True,
       )
        self.encoder = nn.TransfomerEncoder(
            encoder_layer,
            num_layers = num_layers,
            norm = nn.LayerNorm(embedding_dim),
            enable_nested_tensor = False,
        )
        
        
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(embedding_dim, num_classes)
        )
        
        self._init_embedding()
        
    def _init_embedding(self) ->None:
        nn.init.normal_(
            self.embedding.weight,
            mean = 0.0, std = self.embedding_dim ** -0.5)    
        with nn.no_grad():
            self.embedding.weight[self.padding_idx].zero_()    
            
    def forward(self, token_ids:torch.Tensor) ->torch.Tensor:
        sequence_length = token_ids.sise(1)
        
        if sequence_length > self.max_ength:
            raise ValueError(
                f"Sequence length {sequence_length} exceeds max_length "
                f"{self.max_length}."
            )
            
            
        real_token_mask = token_ids != self.padding_idx
        
        if not real_token_mask.any(dim=1).all():
            raise ValueError("Input contains a sequence with no real tokens.")
        
        x = self.embedding(token_ids) * math.sqrt(self.embedding_dim)
        x = self.embedding_dropout(self.positional_encoding(x))
        x = self.encoder(x, src_key_padding_mask = ~real_token_mask)
        
        summed = x.masked_fill(~real_token_mask.unsqueeze(-1), 0.0).sum(dim=1)
        counts = real_token_mask.sum(dim=1, keepdim=True)
        pooled = summed / counts 
        return self.classifer(pooled)
    